In [1]:
# The imports

from dotenv import load_dotenv
from agents.mcp import MCPServerStdio
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from openai import AsyncOpenAI
import os

In [2]:
load_dotenv(override=True)

True

In [11]:
params = {"command": "uv", "args": ["run", "math_server.py"]}
#params = {"command": "python", "args": ["math_server.py"]}

async with MCPServerStdio(params=params, client_session_timeout_seconds=20) as client:
    tools = await client.list_tools()
    resources = await client.session.list_resources()

print(tools)
print()
print(resources)

[Tool(name='add', title=None, description='\n    Takes two numbers, adds them and returns the result of the addition\n    Args:\n        x: An integer\n        y: An integer\n    ', inputSchema={'properties': {'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}, 'required': ['x', 'y'], 'title': 'addArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'integer'}}, 'required': ['result'], 'title': 'addOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None), Tool(name='sub', title=None, description='\n    Takes two numbers(x, y), and returns the result of the subtraction(x-y)\n    Args:\n        x: An integer\n        y: An integer\n    ', inputSchema={'properties': {'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}, 'required': ['x', 'y'], 'title': 'subArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'integer'

In [19]:
instructions = """
You are a mathematician who can solve arithmetic problems intelligently and also tell math jokes. 
when asked to tell a joke, use resources from mcp server.
"""

client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))
model = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=client)

async with MCPServerStdio(params=params, client_session_timeout_seconds=20) as server:
    agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model=model,
            mcp_servers=[server]
            )
    
    with trace("Mathematician"):
            result = await Runner.run(agent, "what is 2 + 3 + 7")
            print(result.final_output)

The result of \( 2 + 3 + 7 \) is \( 12 \).
